# Lab03F — Assimetria agregada e robustez após os pareceres

**Execute tudo e salve uma cópia deste notebook.** Não repetir E, transferência ou laboratórios anteriores. CPU; não precisa GPU. Reutiliza somente o ZIP X autenticado do Drive, sem baixar cotações.

Executa 1.939 ajustes (sete especificações, três recortes), 4.999 reamostragens por bloco5/21/63, média/mediana do universo fixo, seis testes e verificação independente. Preserva80/117 ativos, classes, dez setores, pares e NaNs. São retornos logarítmicos, não preços.

A preparação local já foi calculada e conferida. Esta execução será comparada automaticamente às tabelas e a todos os sorteios locais. A homologação econômica global não foi concluída; o novo bootstrap é aproximado, sem garantia finita de cobertura.


In [ ]:
import os,sys,json,hashlib,zipfile,subprocess,shutil,importlib.util
from pathlib import Path
from datetime import datetime,timezone
from urllib.request import urlretrieve
for module,package in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('matplotlib','matplotlib'),('threadpoolctl','threadpoolctl')]:
    if importlib.util.find_spec(module) is None:subprocess.run([sys.executable,'-m','pip','install','--quiet',package],check=True)
import numpy as np,pandas as pd
try:
    from IPython.display import display,Image
except ImportError:
    def display(value):print(str(value)[:6000])
    def Image(filename):
        from PIL import Image as PillowImage
        im=PillowImage.open(filename);im.load();return filename
LOCAL_MODE=bool(os.environ.get('ALEM_BETA_LOCAL'))
os.environ['ALEM_BETA_EXECUTION_ENV']='local_notebook_integration' if LOCAL_MODE else 'Colab'
BASE=Path(os.environ.get('ALEM_BETA_LOCAL','/content/alem_beta_lab03f')).resolve();BASE.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN=BASE/('run_'+STAMP);RUN.mkdir();ROOT=RUN/'runtime';ROOT.mkdir()
def digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def script(name,*args):subprocess.run([sys.executable,str(ROOT/'scripts'/name),*map(str,args)],check=True)
print('Lab03F | CPU |', 'integração local, não Colab do autor' if LOCAL_MODE else 'Colab', '|',RUN.name)


## Fontes e entradas fixadas

Código e referências compactas vêm de um commit público imutável e são autenticados individualmente. O ZIP de retornos continua externo no Drive. Nenhum download de Yahoo/B3 é executado.


In [ ]:
SOURCE_COMMIT='57727d3eee5796f51b0f0c4b297f91ba537a852e'
SOURCE_PINS={'src/lab02.py': 'f83bef7d5da251fb2571699572f202ee16a8b007522f3e5f1f6f1385f8540b0f', 'src/lab03f.py': 'e65dc527403d5a444b7b52caa5bfd0bde341a48a7b3b57979990c422478c68b7', 'scripts/run_lab03f.py': '0c166c25dc741868dbf1ebc968da525df3148c82685abc8e77a5a86b0d3a6b36', 'scripts/verify_lab03f.py': '9c32ccf39338f75516b2c3b2c13e6a4b246942f7c0637be856954a96d02447eb', 'scripts/plot_lab03f.py': 'ca224ecb8a66f160bafa4b809580b48e07a88ef62cf46ca623d958ea0d08c29f', 'tests/test_lab03f.py': 'f0e45ff4a85bef4508d1d6f78cf173f3525f648a23dfe5a4957709a39c5ae8c8', 'data/lab03f/plano_pos_parecer_v1.json': 'f94cac62a6aa03249fb102ae7f74d5f45f3c28ca4968e503899f11b07cff6d1f', 'data/lab03f/freeze.json': '7b69a88fac384f9a0d4515c0093407966eadec5abf848250c6eafa8bbe6d0388', 'results/tables/lab03b_colab_20261023/asset_betas.csv.gz': '0c2c54fc1f212d914cf083300757e2706a94d32ae762e64b8814e1a837192275', 'results/tables/lab03e_colab_20261023/economic_ledger.csv.gz': 'd095b6be7330ce8f5dc28b425a30b90ab70d1f90c1d63e80d424ab81e634e56e', 'results/tables/lab03e_colab_20261023/economic_sensitivities.csv.gz': '90ebc8dc0a71f65f0a41ebff6afe2d9deea6ffe9d18d57d7e86df4ba0a54fb11', 'results/tables/lab03f_pos_parecer_20261024/individual.csv.gz': 'd1e89113e8d8850563dd14713d67cb8cf026bcc78e1b70f517a70190e7815151', 'results/tables/lab03f_pos_parecer_20261024/aggregate.csv.gz': 'e4f1a4ff05ffc17194e826cd412eb651fe159a3055845cdcc62d7e08569e001a', 'results/tables/lab03f_pos_parecer_20261024/profile_summary.csv': '1560a809f904e4f6c49ac2e5af9767f5b883c350d5278fddfa3895592190f2c4', 'results/tables/lab03f_pos_parecer_20261024/bootstrap_targets.npz': 'c9447e814c9aafc732a029978379f14a837ce950dd3b6ca5e04c12d19e55c1c1'}
for name,expected in SOURCE_PINS.items():
    target=ROOT/name;target.parent.mkdir(parents=True,exist_ok=True)
    if LOCAL_MODE:shutil.copy2(Path(os.environ['ALEM_BETA_LOCAL_SOURCE'])/name,target)
    else:urlretrieve('https://raw.githubusercontent.com/robsondf/alem-do-beta-gemini/'+SOURCE_COMMIT+'/'+name,target)
    assert digest(target)==expected,'Fonte divergente: '+name
print('Fontes autenticadas:',len(SOURCE_PINS),'| commit',SOURCE_COMMIT)
X_ID='1aExT7v-GD8s5xKzM0K-0LIIJER8E8Twg'
X_SHA='297da2afde534082c398324f308df7af134c9692e7d61d6e0b2b820341ea5214'
if LOCAL_MODE:
    X_ARCHIVE=Path(os.environ['ALEM_BETA_LOCAL_X'])
else:
    from google.colab import auth,drive
    auth.authenticate_user();drive.mount('/content/drive')
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    credentials,_=google.auth.default();service=build('drive','v3',credentials=credentials,cache_discovery=False)
    CACHE=Path('/content/drive/MyDrive/alem_do_beta/lab03f_cache');CACHE.mkdir(parents=True,exist_ok=True)
    X_ARCHIVE=CACHE/(X_ID+'.zip')
    if not X_ARCHIVE.exists() or digest(X_ARCHIVE)!=X_SHA:
        name=service.files().get(fileId=X_ID,fields='name').execute()['name']
        found=None
        for candidate in Path('/content/drive/MyDrive').rglob(name):
            if candidate.is_file() and digest(candidate)==X_SHA:found=candidate;break
        if found is not None:X_ARCHIVE=found
        else:
            part=X_ARCHIVE.with_suffix('.part')
            with part.open('wb') as stream:
                download=MediaIoBaseDownload(stream,service.files().get_media(fileId=X_ID));done=False
                while not done:_,done=download.next_chunk()
            assert digest(part)==X_SHA;part.replace(X_ARCHIVE)
assert digest(X_ARCHIVE)==X_SHA
print('Cache X autenticado:',X_ARCHIVE.name,'|',X_ARCHIVE.stat().st_size,'bytes')


## Protocolo e processamento completo

Plano posterior aos pareceres, escrito antes dos novos ajustes, com resultados anteriores conhecidos. Mantém a referência197/474. Bloco21 e Holm10 são a análise agregada principal; blocos5/63 e Holm30 são sensibilidades, todas reportadas. As exclusões de2020 mudam a janela-alvo. As somas de coeficientes dinâmicos são outros estimandos, não prova de correção de assincronia. O avanço é somente histórico, sem uso em previsão.


In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(ROOT/'tests'),'-p','test_lab03f.py','-v'],check=True)
OUTPUT=RUN/'outputs'
script('run_lab03f.py','--x-archive',X_ARCHIVE,'--output',OUTPUT)


In [ ]:
script('verify_lab03f.py','--x-archive',X_ARCHIVE,'--output',OUTPUT)
script('plot_lab03f.py','--output',OUTPUT)
reference=ROOT/'results/tables/lab03f_pos_parecer_20261024'
compared=0
for name in ['individual.csv.gz','aggregate.csv.gz','profile_summary.csv']:
    observed=pd.read_csv(OUTPUT/name,float_precision='round_trip');expected=pd.read_csv(reference/name,float_precision='round_trip')
    pd.testing.assert_frame_equal(observed,expected,check_exact=False,atol=1e-10,rtol=1e-9);compared+=observed.shape[0]*observed.shape[1]
with np.load(OUTPUT/'bootstrap_targets.npz') as actual,np.load(reference/'bootstrap_targets.npz') as expected:
    assert actual.files==expected.files
    for key in actual.files:np.testing.assert_allclose(actual[key],expected[key],atol=1e-10,rtol=1e-9);compared+=actual[key].size
for name,expected in SOURCE_PINS.items():assert digest(ROOT/name)==expected,name
record={'passed':True,'environment':'local_not_author_Colab' if LOCAL_MODE else 'Colab','source_commit':SOURCE_COMMIT,'source_files':len(SOURCE_PINS),'compared_values':compared,'atol':1e-10,'rtol':1e-9,'X_sha256':digest(X_ARCHIVE),'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__}
(OUTPUT/'notebook_comparison.json').write_text(json.dumps(record,indent=2)+'\n')
print(json.dumps(record,indent=2))


In [ ]:
aggregate=pd.read_csv(OUTPUT/'aggregate.csv.gz')
display(aggregate[aggregate.block.eq(21)][['profile','scenario','statistic','estimate','ci_low','ci_high','p_holm10','p_holm30']])
display(pd.read_csv(OUTPUT/'profile_summary.csv'))
for name in ['lab03f_agregado','lab03f_criticos']:display(Image(filename=str(OUTPUT/'figures'/(name+'.png'))))


## Interpretação e salvamento

O novo resultado agrega evidência sobre a média da década, com sensibilidade à família e à janela. Não demonstra assimetria generalizada. As duas rejeições individuais originais são frágeis às exclusões de crise; não se escolhe outra especificação por rejeitar outro ativo. Os recortes recentes permanecem inconclusivos.

ICs bootstrap são marginais/aproximados, sujeitos a estacionariedade local e regularidade; ajuste Holm não corrige valores-p descalibrados. Máscaras de influência são após seleção; seu Holm não garante FWER. Nada disso elimina a auditoria econômica incompleta, sobrevivência/vintages ou lacunas não identificáveis. A representação/raciocínio continua proposta separada.

O ZIP final salva fontes necessárias, resultados, gráficos, hashes e comparação; não contém a matriz privada de retornos. A referência ao cache externo é mantida por ID/SHA. Depois de executar tudo, **salve também o notebook**.


In [ ]:
export=RUN/('resultados_lab03f_'+STAMP+'.zip')
inventory={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for base,prefix in [(ROOT,'runtime'),(OUTPUT,'outputs')]:
        for path in sorted(base.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                name=prefix+'/'+str(path.relative_to(base));z.write(path,name);inventory[name]=digest(path)
    z.writestr('manifesto_exports.json',json.dumps(inventory,indent=2))
print('Export:',export.name,'| bytes:',export.stat().st_size,'| SHA256:',digest(export))
if not LOCAL_MODE:
    folder=Path('/content/drive/MyDrive/alem_do_beta/lab03f_resultados');folder.mkdir(parents=True,exist_ok=True)
    destination=folder/export.name;shutil.copy2(export,destination);assert digest(destination)==digest(export)
    print('SALVO NO DRIVE:',destination)
else:print('Integração local concluída. Não é execução Colab do autor.')
